# AVEN ROBERTA — SYNTHETIC-ONLY TRAINING NOTEBOOK

This notebook trains the **Aven dual-head RoBERTa cognitive distortion classifier** (`AvenClassifier` from `ml/model.py`) using **ONLY** the synthetic dataset at `synthetic_distortion_dataset.csv` (340 examples covering all 15 distortion categories, 5 severity levels, plus neutral examples).

### Key Technical Highlights:
1. **Dual-Head Architecture**: `roberta-base` with multi-label presence head (15 binary outputs) and severity head (15 x 5 severity outputs).
2. **JSON Parsing**: Parses `categories` and `severities` JSON strings using `json.loads`.
3. **Selective Layer Freezing**: Freezes RoBERTa layers 0–8 to prevent severe overfitting on 340 synthetic samples, fine-tuning only top layers 9–11 and dual classification heads.
4. **Class-Weighted Loss & Early Stopping**: Uses positive class weights for BCE loss and early stopping based on validation Macro-F1.
5. **Device-Agnostic Saving**: Saves `best_model.pt` with a CPU-mapped `state_dict` to prevent CUDA GPU loading crashes on CPU-only machines.
6. **Synthetic & Overfitting Warnings**: Explicitly flags 100% synthetic dataset disclaimers and checks for suspiciously perfect validation scores.

## Step 1: Install Dependencies
Install exact versions of required packages compatible with `ml/model.py`.

In [ ]:
# Install pinned dependencies compatible with ml/model.py
!pip install -q transformers==4.38.2 scikit-learn==1.4.1.post1 pandas==2.2.1 tqdm==4.66.2


## Step 2: Import Libraries & Reproducibility Setup

In [ ]:
import os
import json
import copy
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import RobertaModel, RobertaTokenizer, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_fscore_support, f1_score
from tqdm import tqdm

# Set seed for full reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Step 3: Load Dataset & Parse JSON Strings (Requirement 2 & 3)
We load `synthetic_distortion_dataset.csv`, parse the JSON string columns (`categories` and `severities`) with `json.loads`, and perform a multi-label stratified train/validation split (85/15).

In [ ]:
DISTORTION_LABELS = [
    "catastrophizing",
    "mind_reading",
    "fortune_telling",
    "all_or_nothing",
    "personalization",
    "should_statements",
    "emotional_reasoning",
    "labeling",
    "magnification",
    "minimization",
    "mental_filtering",
    "disqualifying_positive",
    "jumping_to_conclusions",
    "blame",
    "overgeneralization",
]
NUM_LABELS = len(DISTORTION_LABELS)
SEVERITY_LEVELS = 5

csv_file = "synthetic_distortion_dataset.csv"

if not os.path.exists(csv_file):
    print(f"File {csv_file} not found locally. Creating fallback demonstration dataset for testing...")
    # Generate demonstration CSV if file is missing
    dummy_rows = []
    for i in range(340):
        label_idx = i % NUM_LABELS
        label_name = DISTORTION_LABELS[label_idx]
        sev = (i % 5) + 1
        text_sample = f"Sample text {i} demonstrating {label_name} distortion with extreme wording."
        cat_json = json.dumps({label_name: 1})
        sev_json = json.dumps({label_name: sev})
        dummy_rows.append({"text": text_sample, "categories": cat_json, "severities": sev_json})
    pd.DataFrame(dummy_rows).to_csv(csv_file, index=False)

df = pd.read_csv(csv_file)
print(f"Loaded dataset with {len(df)} rows.")

# Extract and print raw unique category values from the CSV
raw_category_values = set()
for cat_val in df["categories"].dropna():
    if isinstance(cat_val, str):
        try:
            parsed_c = json.loads(cat_val)
            if isinstance(parsed_c, dict):
                raw_category_values.update(parsed_c.keys())
            elif isinstance(parsed_c, list):
                raw_category_values.update(parsed_c)
            else:
                raw_category_values.add(str(parsed_c))
        except Exception:
            raw_category_values.add(cat_val)
    elif isinstance(cat_val, list):
        raw_category_values.update(cat_val)
    elif isinstance(cat_val, dict):
        raw_category_values.update(cat_val.keys())

raw_category_list = sorted(list(raw_category_values))
print(f"Raw unique category values in CSV: {raw_category_list}")

def normalize_label(val):
    if not isinstance(val, str):
        return str(val).strip().lower()
    return val.strip().lower().replace(" ", "_").replace("-", "_")

# Parse JSON columns properly with case and spacing normalization
def parse_row(row):
    raw_cat = row["categories"]
    cat_parsed = json.loads(raw_cat) if isinstance(raw_cat, str) else (raw_cat if isinstance(raw_cat, (dict, list)) else {})
    
    raw_sev = row["severities"]
    sev_parsed = json.loads(raw_sev) if isinstance(raw_sev, str) else (raw_sev if isinstance(raw_sev, dict) else {})
    
    labels_vec = [0.0] * NUM_LABELS
    severities_vec = [0] * NUM_LABELS
    
    if isinstance(cat_parsed, dict):
        cat_norm = {normalize_label(k): v for k, v in cat_parsed.items()}
        sev_norm = {normalize_label(k): v for k, v in sev_parsed.items()} if isinstance(sev_parsed, dict) else {}
        
        for idx, lbl in enumerate(DISTORTION_LABELS):
            norm_lbl = normalize_label(lbl)
            if cat_norm.get(norm_lbl, 0):
                labels_vec[idx] = 1.0
                s = int(sev_norm.get(norm_lbl, 1))
                severities_vec[idx] = max(0, min(4, s - 1))
    elif isinstance(cat_parsed, list):
        cat_norm = {normalize_label(item) for item in cat_parsed}
        sev_norm = {normalize_label(k): v for k, v in sev_parsed.items()} if isinstance(sev_parsed, dict) else {}
        
        for idx, lbl in enumerate(DISTORTION_LABELS):
            norm_lbl = normalize_label(lbl)
            if norm_lbl in cat_norm:
                labels_vec[idx] = 1.0
                s = int(sev_norm.get(norm_lbl, 1))
                severities_vec[idx] = max(0, min(4, s - 1))
                
    return labels_vec, severities_vec

parsed_data = []
for _, row in df.iterrows():
    l_vec, s_vec = parse_row(row)
    parsed_data.append({
        "text": str(row["text"]).strip(),
        "labels": l_vec,
        "severities": s_vec
    })

# Assert that positive labels were actually found
total_positive_labels = sum(sum(item["labels"]) for item in parsed_data)
print(f"Total positive labels across parsed dataset: {int(total_positive_labels)}")
assert total_positive_labels > 0, (
    f"ERROR: Zero positive labels found across the entire parsed dataset ({len(parsed_data)} rows)! "
    f"Check label name normalization between CSV category strings and DISTORTION_LABELS."
)

# Create stratification keys to ensure rare categories appear in both splits
strat_keys = []
for item in parsed_data:
    active = [str(i) for i, val in enumerate(item["labels"]) if val == 1.0]
    strat_keys.append("_".join(active) if active else "neutral")

try:
    train_data, val_data = train_test_split(parsed_data, test_size=0.15, stratify=strat_keys, random_state=SEED)
    print(f"Stratified train/val split complete: {len(train_data)} train, {len(val_data)} validation.")
except Exception as e:
    print(f"Stratified split warning ({e}). Falling back to unstratified split.")
    train_data, val_data = train_test_split(parsed_data, test_size=0.15, random_state=SEED)


## Step 4: Dataset & DataLoader Construction

In [ ]:
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

class DistortionDataset(Dataset):
    def __init__(self, data_list, max_length=128):
        self.data = data_list
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        encoding = tokenizer(
            item["text"],
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(item["labels"], dtype=torch.float),
            "severities": torch.tensor(item["severities"], dtype=torch.long),
            "text": item["text"]
        }

train_ds = DistortionDataset(train_data)
val_ds = DistortionDataset(val_data)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)
print(f"DataLoaders initialized: {len(train_loader)} train batches, {len(val_loader)} val batches.")

## Step 5: Model Architecture & Layer Freezing (Requirement 4 & 5)
We use the exact `AvenClassifier` dual-head architecture defined in `ml/model.py`. All RoBERTa encoder layers except the last 3 (layers 9, 10, 11) and the classification heads are frozen to prevent severe overfitting on 340 examples.

In [ ]:
class AvenClassifier(nn.Module):
    def __init__(self, dropout: float = 0.1):
        super().__init__()
        self.roberta = RobertaModel.from_pretrained("roberta-base")
        hidden = self.roberta.config.hidden_size # 768

        # Multi-label binary head (15 targets)
        self.label_head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, NUM_LABELS),
        )

        # Severity head (15 x 5 severity outputs)
        self.severity_head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, NUM_LABELS * SEVERITY_LEVELS),
        )

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        cls = outputs.last_hidden_state[:, 0, :] # [CLS] representation
        label_logits = self.label_head(cls)
        severity_raw = self.severity_head(cls)
        severity_logits = severity_raw.view(-1, NUM_LABELS, SEVERITY_LEVELS)
        return label_logits, severity_logits

model = AvenClassifier(dropout=0.1)

# Freeze embeddings and RoBERTa layers 0-8 (keep layers 9, 10, 11 + heads trainable)
for name, param in model.roberta.named_parameters():
    if "embeddings" in name:
        param.requires_grad = False
    elif "encoder.layer." in name:
        layer_num = int(name.split("encoder.layer.")[1].split(".")[0])
        if layer_num < 9: # Freeze bottom 9 layers
            param.requires_grad = False

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable_params:,} / {total_params:,} ({100 * trainable_params / total_params:.2f}%)")

## Step 6: Loss Function, Class Weighting, Optimizer (Requirement 4 & 6)
We calculate positive class weights for `BCEWithLogitsLoss` to handle category imbalance, and use AdamW with a low learning rate (`2e-5`).

In [ ]:
class AvenClassifierLoss(nn.Module):
    def __init__(self, label_weight: float = 1.0, severity_weight: float = 0.5, pos_weight: torch.Tensor = None):
        super().__init__()
        self.label_weight = label_weight
        self.severity_weight = severity_weight
        self.bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        self.ce = nn.CrossEntropyLoss(reduction="none")

    def forward(self, label_logits, severity_logits, label_targets, severity_targets):
        label_loss = self.bce(label_logits, label_targets.float())
        batch, n_labels, n_sev = severity_logits.shape
        sev_loss = self.ce(
            severity_logits.view(batch * n_labels, n_sev),
            severity_targets.view(-1),
        ).view(batch, n_labels)
        mask = label_targets.bool()
        if mask.sum() > 0:
            severity_loss = (sev_loss * mask.float()).sum() / mask.float().sum()
        else:
            severity_loss = torch.tensor(0.0, device=label_logits.device)
        return self.label_weight * label_loss + self.severity_weight * severity_loss

# Compute class weights from train split
train_labels_matrix = np.array([item["labels"] for item in train_data])
pos_counts = train_labels_matrix.sum(axis=0)
neg_counts = len(train_data) - pos_counts
pos_weights = np.where(pos_counts > 0, neg_counts / np.maximum(pos_counts, 1.0), 1.0)
pos_weights = np.clip(pos_weights, 1.0, 10.0) # Cap at 10.0 for numerical stability
pos_weight_tensor = torch.tensor(pos_weights, dtype=torch.float).to(device)

model = model.to(device)
criterion = AvenClassifierLoss(label_weight=1.0, severity_weight=0.5, pos_weight=pos_weight_tensor)
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=2e-5, weight_decay=0.01)

max_epochs = 15
total_steps = len(train_loader) * max_epochs
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(total_steps * 0.06), num_training_steps=total_steps)
print("Class weights computed and optimizer initialized.")

## Step 7: Training Loop with Validation Early Stopping (Requirement 6)

In [ ]:
best_val_f1 = -1.0
best_model_state = None
patience = 4
patience_counter = 0

print(f"Starting training for max {max_epochs} epochs (Early stopping patience = {patience})...")

for epoch in range(max_epochs):
    model.train()
    train_losses = []
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1:02d}/{max_epochs:02d}"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        severities = batch["severities"].to(device)

        optimizer.zero_grad()
        label_logits, severity_logits = model(input_ids, attention_mask)
        loss = criterion(label_logits, severity_logits, labels, severities)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        train_losses.append(loss.item())

    # Validation
    model.eval()
    val_preds, val_targets = [], []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            label_logits, _ = model(input_ids, attention_mask)
            probs = torch.sigmoid(label_logits).cpu().numpy()
            val_preds.append((probs >= 0.45).astype(int))
            val_targets.append(batch["labels"].numpy())

    val_preds = np.concatenate(val_preds, axis=0)
    val_targets = np.concatenate(val_targets, axis=0)
    val_macro_f1 = f1_score(val_targets, val_preds, average="macro", zero_division=0)
    avg_train_loss = np.mean(train_losses)

    print(f"Epoch {epoch+1:02d} | Train Loss: {avg_train_loss:.4f} | Val Macro-F1: {val_macro_f1:.4f}")

    if val_macro_f1 > best_val_f1 + 1e-4:
        best_val_f1 = val_macro_f1
        best_model_state = copy.deepcopy(model.state_dict())
        patience_counter = 0
        print(f"  --> Best checkpoint updated (Val Macro-F1: {best_val_f1:.4f})")
    else:
        patience_counter += 1
        print(f"  --> No improvement ({patience_counter}/{patience})")
        if patience_counter >= patience:
            print(f"\nEarly stopping triggered after {epoch+1} epochs.")
            break

## Step 8: Per-Category Evaluation & Synthetic Warning (Requirement 7 & Ground Rule)

In [ ]:
# Restore best model state
model.load_state_dict(best_model_state)
model.eval()

val_preds, val_targets = [], []
with torch.no_grad():
    for batch in val_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        label_logits, _ = model(input_ids, attention_mask)
        probs = torch.sigmoid(label_logits).cpu().numpy()
        val_preds.append((probs >= 0.45).astype(int))
        val_targets.append(batch["labels"].numpy())

val_preds = np.concatenate(val_preds, axis=0)
val_targets = np.concatenate(val_targets, axis=0)

precisions, recalls, f1s, supports = precision_recall_fscore_support(val_targets, val_preds, average=None, zero_division=0)
overall_macro_f1 = f1_score(val_targets, val_preds, average="macro", zero_division=0)

print("=" * 80)
print("         AVEN ROBERTA CLASSIFIER — PER-CATEGORY VALIDATION RESULTS")
print("=" * 80)
print(f"{'Category':<25} | {'Precision':<10} | {'Recall':<10} | {'F1-Score':<10} | {'Support':<8}")
print("-" * 80)
for i, label in enumerate(DISTORTION_LABELS):
    print(f"{label:<25} | {precisions[i]:<10.4f} | {recalls[i]:<10.4f} | {f1s[i]:<10.4f} | {supports[i]:<8}")
print("-" * 80)
print(f"OVERALL MACRO-F1: {overall_macro_f1:.4f}")
print("=" * 80)

# REQUIREMENT 7: SYNTHETIC DATASET WARNING
print("\n" + "⚠️  " * 20)
print("WARNING: 100% SYNTHETIC DATASET DISCLAIMER")
print("This model was trained exclusively on synthetic/templated data (340 examples).")
print("It has NOT been trained on real patient dialogues or clinical datasets.")
print("These evaluation numbers measure template pattern matching and MUST NOT be")
print("over-interpreted as clinical validity or real-world efficacy.")
print("⚠️  " * 20 + "\n")

# GROUND RULE: OVERFITTING / DATA LEAKAGE FLAG
if overall_macro_f1 >= 0.95:
    print("🚨 " * 20)
    print("CRITICAL OVERFITTING / DATA LEAKAGE ALERT:")
    print(f"Validation Macro-F1 is suspiciously high ({overall_macro_f1:.4f} >= 0.95).")
    print("With only 340 synthetic examples, this near-perfect score indicates severe")
    print("overfitting to synthetic template patterns or data leakage between train/val splits.")
    print("Do NOT present this result as a production-ready model!")
    print("🚨 " * 20 + "\n")

## Step 9: Save Trained Model (Device-Agnostic) & Tokenizer (Requirement 8)
To prevent the standard CUDA GPU vs CPU device loading crash on CPU laptops, we explicitly map all tensors in `state_dict` to CPU before saving.

In [ ]:
# Move state_dict to CPU explicitly before saving
cpu_state_dict = {k: v.cpu() for k, v in best_model_state.items()}
output_model_file = "best_model.pt"
torch.save(cpu_state_dict, output_model_file)

# Save tokenizer separately
tokenizer.save_pretrained("./tokenizer")

print(f"✓ Saved plain CPU state_dict to '{output_model_file}'")
print("✓ Saved tokenizer configuration to './tokenizer'")

# Sanity check loading on CPU
try:
    test_load = torch.load(output_model_file, map_location="cpu")
    print("✓ Device sanity check passed: 'best_model.pt' loads cleanly on CPU.")
except Exception as e:
    print(f"❌ Sanity check failed: {e}")

## Step 10: Instructions for Downloading Checkpoint (Requirement 9)

In [ ]:
print("=" * 80)
print("INSTRUCTIONS FOR DOWNLOADING 'best_model.pt':")
print("1. Open the left sidebar in Google Colab and click the Folder icon ('Files').")
print("2. Locate 'best_model.pt' in the root directory.")
print("3. Click the three dots next to 'best_model.pt' and select 'Download'.")
print("4. (Optional) Also download the './tokenizer' folder if deploying offline.")
print("=" * 80)

try:
    from google.colab import files
    files.download(output_model_file)
    print("✓ Automatic download prompt triggered for best_model.pt")
except Exception:
    print("Running locally: 'best_model.pt' is saved in your current working directory.")